In [ ]:
# AttentionPhi (Transformer) visualization
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
import torchvision.transforms as transforms

from models.attention_phi import AttentionPhi
from models.dino_attention import DinoAttentionExtractor
from models import networks

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DINO_SIZE = 224
IMG_SIZE = 224

# Phi configuration (Matching train.py args)
PHI_INPUT_DOMAIN = "attention"  # "attention" or "feature"
PHI_HIDDEN_CHANNELS = 256        # d_model
PHI_NUM_LAYERS = 4
PHI_FEATURE_DIM = 768

EXP_NAME = "phi_transformer_native_v1"
CHECKPOINT_DIR = Path(f"/home/ljc/code/PaBoT-main/checkpoints/{EXP_NAME}")

# Target images for visualization
IMG_PATH_CT = "/home/ljc/code/PaBoT-main/datasets/testA/1BA278_slice192.png"
IMG_PATH_MRI = "/home/ljc/code/PaBoT-main/datasets/testB/1BA278_slice192.png"

def preprocess_image(image_path, image_size=224):
    # Standard normalization for DINO
    transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ])
    img = Image.open(image_path).convert("RGB")
    return transform(img).unsqueeze(0)

def normalize_to_01(feat, eps=1e-6):
    min_v = feat.amin(dim=(2, 3), keepdim=True)
    max_v = feat.amax(dim=(2, 3), keepdim=True)
    return (feat - min_v) / (max_v - min_v + eps)

def load_phi_checkpoint(ckpt_path, device):
    phi_in_channels = 1 if PHI_INPUT_DOMAIN == "attention" else (1 + PHI_FEATURE_DIM)
    netPhi = AttentionPhi(
        in_channels=phi_in_channels, 
        d_model=PHI_HIDDEN_CHANNELS, 
        num_layers=PHI_NUM_LAYERS
    ).to(device).eval()
    
    state = torch.load(ckpt_path, map_location=device)
    # Handle DataParallel prefix if necessary
    if any(k.startswith('module.') for k in state.keys()):
        state = {k.replace('module.', ''): v for k, v in state.items()}
        
    netPhi.load_state_dict(state, strict=True)
    return netPhi

print("Loading DINO and images...")
dino_extractor = DinoAttentionExtractor(model_name="dino_vitb8", image_size=DINO_SIZE).to(DEVICE).eval()
img_ct = preprocess_image(IMG_PATH_CT, IMG_SIZE).to(DEVICE)
img_mri = preprocess_image(IMG_PATH_MRI, IMG_SIZE).to(DEVICE)

# List of checkpoints to visualize
# You can use ["best_phi"] or [10, 20, 50, "latest"]
viz_epochs = ["best_phi", "latest"]

with torch.no_grad():
    # Get Ground Truth CT Attention
    attn_ct_real = dino_extractor(img_ct)
    attn_ct_real_vis = normalize_to_01(attn_ct_real)[0, 0].cpu().numpy()
    
    # Get MRI DINO features
    need_feat = (PHI_INPUT_DOMAIN == "feature")
    out_mri = dino_extractor(img_mri, return_cls_attn=True, return_patch_feat=need_feat)
    attn_mri = out_mri[0]
    feat_mri = out_mri[2] if need_feat else None
    attn_mri_vis = normalize_to_01(attn_mri)[0, 0].cpu().numpy()

rows = len(viz_epochs)
fig, axes = plt.subplots(rows, 3, figsize=(15, 5 * rows))
if rows == 1:
    axes = np.expand_dims(axes, axis=0)

for row, epoch in enumerate(viz_epochs):
    ckpt_path = CHECKPOINT_DIR / f"{epoch}_net_Phi.pth"
    if not ckpt_path.exists():
        print(f"[skip] missing: {ckpt_path}")
        continue

    print(f"[load] {ckpt_path.name}")
    netPhi = load_phi_checkpoint(str(ckpt_path), DEVICE)

    with torch.no_grad():
        # Construct Phi input
        if PHI_INPUT_DOMAIN == "feature":
            # Simple interpolation to match spatial size if needed
            if feat_mri.shape[-2:] != attn_mri.shape[-2:]:
                feat_mri = F.interpolate(feat_mri, size=attn_mri.shape[-2:], mode="bilinear")
            phi_input = torch.cat([feat_mri, attn_mri], dim=1)
        else:
            phi_input = attn_mri
            
        fake_attn_ct = netPhi(phi_input)
        fake_attn_ct_vis = normalize_to_01(fake_attn_ct)[0, 0].cpu().numpy()

    axes[row, 0].imshow(attn_mri_vis, cmap="jet")
    axes[row, 0].set_title(f"MRI Attention (DINO)")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(fake_attn_ct_vis, cmap="jet")
    axes[row, 1].set_title(f"Predicted CT Attention ({epoch})")
    axes[row, 1].axis("off")

    axes[row, 2].imshow(attn_ct_real_vis, cmap="jet")
    axes[row, 2].set_title("Real CT Attention (DINO)")
    axes[row, 2].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# Single checkpoint visualization by specific path
SPECIFIC_CKPT_PATH = "/home/sun_yuxi/luo_jiacheng/pabot/pabot/checkpoints/phi_transformer_native_v1/176_net_Phi.pth"

if Path(SPECIFIC_CKPT_PATH).exists():
    print(f"[load] {SPECIFIC_CKPT_PATH}")
    netPhi_single = load_phi_checkpoint(SPECIFIC_CKPT_PATH, DEVICE)
    
    with torch.no_grad():
        # Construct input
        if PHI_INPUT_DOMAIN == "feature":
            phi_input_single = torch.cat([feat_mri, attn_mri], dim=1)
        else:
            phi_input_single = attn_mri
            
        fake_attn_ct_single = netPhi_single(phi_input_single)
        fake_attn_ct_vis_single = normalize_to_01(fake_attn_ct_single)[0, 0].cpu().numpy()
        
    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].imshow(attn_mri_vis, cmap="jet")
    axes[0].set_title("MRI Attention (DINO)")
    axes[0].axis("off")

    axes[1].imshow(fake_attn_ct_vis_single, cmap="jet")
    axes[1].set_title(f"Predicted CT Attention\n({Path(SPECIFIC_CKPT_PATH).name})")
    axes[1].axis("off")

    axes[2].imshow(attn_ct_real_vis, cmap="jet")
    axes[2].set_title("Real CT Attention (DINO)")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()
else:
    print(f"[error] path not found: {SPECIFIC_CKPT_PATH}")
